# Procesamiento 
Unimos los datasets mensuales, realizamos filtro de acuerdo a los IDs de interes (M30), y realizamos limpieza de los datos.

## Paso 1: Filtrado

In [1]:
import pandas as pd

# Definimos la ruta del archivo CSV con datos de sensores
ruta_sensores = "../data/extra/ubicaciones_m30.csv"

# Cargamos el archivo CSV en un DataFrame de pandas
df_sensores = pd.read_csv(ruta_sensores, sep=',', encoding='latin1')

# Extraemos unicamento los IDs pertenecientes a la M30
ids_m30 = df_sensores['id'].unique()

# Validamos los resultados 
print(f"Total de sensores registrados: {len(df_sensores)}")
print(f"Total de sensores M30 aislados: {len(ids_m30)}")
print(f"Muestra de los primeros 5 IDs: {ids_m30[:5]}")

Total de sensores registrados: 353
Total de sensores M30 aislados: 353
Muestra de los primeros 5 IDs: [6668 6669 6688 6691 6692]


## Paso 2 (Procesamiento por lotes) y 3 (Limpieza)
Para evitar un colapso de memoria del ordenador ejecutaremos el procesamiento usando un bucle for, para ir itereando sobre cada fichero(mes), filtramos y realizamos una limpieza de acuerdo con el error.

In [2]:
import glob
import os
from tqdm import tqdm  # Importamos la librería de la barra de progreso

# Definimos la ruta donde estan guardados nuestros csv mensuales 
# Agregamos el *\.csv para que tome todos los archivos 
ruta_csv_mensuales = "../data/raw/*.csv"
#ruta_csv_mensuales = "../data/test/*.csv"

# Obtenemos la lista de todos los archivos 
archivos_csv = glob.glob(ruta_csv_mensuales)
print(f"Archivos mensuales encontrados: {len(archivos_csv)}")

# Creamos una lista vacia para ir guardando los DF filtrados 
datos_m30_procesados = []

# Creamos un bucle para hacer procesamiento por lotes
# Hacemos control de columnas corruptas con on_bad_lines='skip'
# Hacemos low_memory=False analizamos mejor los tipos de datos
# Usamos la libreria tqdm para calcular tiempo de espera del proceso
for archivo in tqdm(archivos_csv, desc="Procesando históricos (1h)", unit="mes"):

    # Leemos el mes actual y hacemos manejo de errores
    try:
        df_mes = pd.read_csv(archivo, sep=';', encoding='utf-8', on_bad_lines='skip', low_memory=False)
    except:
        df_mes = pd.read_csv(archivo, sep=';', encoding='latin1', on_bad_lines='skip', low_memory=False)
        
    # Empezamos proceso de limpieza

    # Nos quedamos solamente con los sensores de la M30
    df_mes_m30 = df_mes[df_mes['id'].isin(ids_m30)].copy()

    # Hacemos eliminacion de valores nulos (NaN) en columnas clave
    # Agregamos variables PRIMER CAMBIO
    columnas_clave = ['id', 'fecha', 'intensidad', 'ocupacion', 'carga', 'vmed', 'error']
    df_mes_limpio = df_mes_m30.dropna(subset=columnas_clave)

    # Eliminamos valores duplicados 
    df_mes_limpio = df_mes_limpio.drop_duplicates(subset=['id', 'fecha'])

    # Filtramos datos de acuerdo a la columna error (error='N')
    # Filtramos valores ausentes o negativos 
    df_mes_limpio = df_mes_limpio[
        (df_mes_limpio['error'] == 'N') &
        (df_mes_limpio['intensidad'] >= 0) &
        (df_mes_limpio['ocupacion'] >= 0) &
        (df_mes_limpio['carga'] >= 0) &
        (df_mes_limpio['vmed'] >= 0)
    ].copy()

    # Hacemos transformacion temporal
    # Transformamos la columna fecha a formato datetime 
    df_mes_limpio['fecha'] = pd.to_datetime(df_mes_limpio['fecha'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

    # Establecemos la fecha como indice
    df_mes_limpio = df_mes_limpio.set_index('fecha')

    # Agrupamos por 'id' y calculamos la media cada hora
    # Usamos la funcion exclusiva de series temporales resample
    df_agrupado = df_mes_limpio.groupby('id').resample('1h').agg({
        'intensidad': 'mean',
        'ocupacion': 'mean',
        'carga': 'mean',
        'vmed': 'mean'
    }).reset_index()

    # Volvemos a eliminar nulos que nos podria generar la funcion resample
    df_agrupado = df_agrupado.dropna(subset=['intensidad', 'vmed'])

    # Almacenamiento local (ahora ocupa un 75% menos de RAM)
    datos_m30_procesados.append(df_agrupado)

print(f"Procesamiento por lotes y limpieza finalizado")

Archivos mensuales encontrados: 50


Procesando históricos (1h): 100%|██████████| 50/50 [08:14<00:00,  9.90s/mes]

Procesamiento por lotes y limpieza finalizado


## Paso 4: Consolidacion
Realizamos la unificacion del dataset, los ordenamos de acuerdo a ID y fecha
Finalmente exportamos en formato parquet

In [3]:
# Concatenamos todos los datasets en uno solo
print("Concatenando todos los meses históricos limpios...")
df_final = pd.concat(datos_m30_procesados, ignore_index=True)

# Ordenamos cronologicamente por id y luego por fecha
print("Ordenando cronológicamente las series temporales...")
df_final = df_final.sort_values(by=['id', 'fecha']).reset_index(drop=True)

# Validamos el tamaño deldataset final y sus fechas
print(f"Total de registros limpios consolidados: {len(df_final)}")
print(f"Fecha mínima: {df_final['fecha'].min()}")
print(f"Fecha máxima: {df_final['fecha'].max()}")
print(f"Total de sensores únicos: {df_final['id'].nunique()}")

# Exportamos en formato parquet 
ruta_salida = "../data/processed/dataset_completo.parquet"
print(f"Exportando a {ruta_salida}...")
df_final.to_parquet(ruta_salida, index=False)

print("\n¡Fase de Data Preparation inicial completada con éxito!")

Concatenando todos los meses históricos limpios...
Ordenando cronológicamente las series temporales...
Total de registros limpios consolidados: 9624596
Fecha mínima: 2022-01-01 00:00:00
Fecha máxima: 2026-02-28 23:00:00
Total de sensores únicos: 350
Exportando a ../data/processed/dataset_completo.parquet...

¡Fase de Data Preparation inicial completada con éxito!


In [4]:
# Identificar que sensores se pierden
sensores_procesados = df_final['id'].unique()
sensores_perdidos = set(ids_m30) - set(sensores_procesados)
print(f"IDs de los sensores perdidos en este periodo: {sensores_perdidos}")

IDs de los sensores perdidos en este periodo: {np.int64(11376), np.int64(11386), np.int64(7118)}


### Por sugerencia del profesor se trabajara solo con 4 dataset, se elimina la variable y se trabajara unicamente con los años desde 2024, por lo cual trabajaremos sobre el mismo parquet que ya procesamos en este notebook, para ahorrar potencia de procesamiento, teniendo en cuenta el volumen de los datos.